# 05 — LLM Agent (Evidence-Traced Graph QA)

**Pipeline Step 5 of 5**

This notebook demonstrates the LLM-powered QA agent that queries the Micro-CKG with strict evidence traceability. Every answer cites exact `(Source)--[Edge_Type, Score=X.XX]-->(Target)` graph evidence.

### Hardened Guardrails
1. **No External Knowledge** — answers from graph context only
2. **Missing Data Fallback** — explicit "No evidence found" response
3. **Mandatory Citation** — `[Evidence: (Source) --(Edge)--> (Target)]`
4. **Objective Tone** — no speculation

### Prerequisites
- Ollama installed: `brew install ollama`
- Model pulled: `ollama pull llama3.1:8b`
- Server running: `ollama serve`

### Inputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG from Step 04 |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.biocypher_adapter import load_graph
from src.llm_agent import create_qa_agent, query_graph

CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 5.1 Load Micro-CKG

We load the persisted GraphML file produced by notebook 04. The graph is deserialized back into a NetworkX DiGraph with all node and edge attributes intact. The summary below confirms the graph structure matches expectations (number of gene nodes, cell-type nodes, region nodes, and total edges).

In [2]:
graph = load_graph(CACHE_DIR / "micro_ckg.graphml")
print(f"\nMicro-CKG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

  Graph loaded: 45 nodes, 338 edges

Micro-CKG: 45 nodes, 338 edges


## 5.2 Create QA Agent (Ollama — Local LLM)

The QA agent uses **Ollama** with `llama3.1:8b` (128K context) for fully local, free inference. The system prompt contains the serialized graph data and strict evidence-traceability rules.

To switch providers, change `provider` to `"google"` or `"openai"` and set the corresponding API key.

In [3]:
agent = create_qa_agent(graph, provider="ollama")
print("QA agent initialised (Ollama — llama3.1:8b).")

QA agent initialised (Ollama — llama3.1:8b).


## 5.3 Killer Demo Queries

Three questions designed to demonstrate the agent's evidence-traceability guardrails:

1. **Biomarker Discovery** — What statistically significant genes drive the AD condition? Tests the agent's ability to cite exact DE evidence from graph edges.
2. **Spatial Context** — Which anatomical regions harbor key neuroinflammatory markers? Tests the agent's ability to traverse gene → cell-type → region paths.
3. **Anti-Hallucination Trap** — Ask about Tau tangles and Lecanemab (neither exists in the graph). Tests the "No evidence found" guardrail.

In [4]:
# Query 1: Biomarker Discovery
question1 = "Based on the Micro-CKG, what are the top statistically significant genes driving the AD (Disease) condition?"
answer1 = query_graph(agent, question1)
print(answer1)

  Querying agent: Based on the Micro-CKG, what are the top statistically significant genes driving...


The Micro-CKG (Microbiome-Connectome-Gut) data provided does not explicitly mention the top statistically significant genes driving the AD (Disease) condition. However, we can infer some information from the data.

The data appears to be a list of gene-cell type associations, where each row represents a gene-cell type pair with a corresponding p-value. To identify the top statistically significant genes driving the AD condition, we would need to filter the data to only include genes that are associated with AD-related cell types (e.g., microglia, astrocytes) and have a low p-value (e.g., < 0.05).

Unfortunately, the data does not provide a clear indication of which genes are associated with AD. However, we can try to identify some genes that are consistently associated with AD-related cell types.

One approach is to look for genes that are associated with multiple AD-related cell types. For example, the gene "Tsc22d4" is associated with microglia (Cluster_0_Cortex), astrocytes (Cluster

In [5]:
# Query 2: Spatial Context
question2 = "Which specific spatial clusters or anatomical regions are these key neuroinflammatory markers (e.g., Fth1, Prnp, Calb1) mapped to in the graph?"
answer2 = query_graph(agent, question2)
print(answer2)

  Querying agent: Which specific spatial clusters or anatomical regions are these key neuroinflamm...
Unfortunately, the provided text does not explicitly mention the specific spatial clusters or anatomical regions that the key neuroinflammatory markers (e.g., Fth1, Prnp, Calb1) are mapped to. The text only shows the associations between genes and cell types, but does not provide information on the spatial clusters or anatomical regions.

However, based on the gene names mentioned (e.g., Fth1, Prnp, Calb1), it is likely that they are related to neuroinflammation or neurodegenerative diseases. Fth1 is involved in iron metabolism, Prnp is associated with prion diseases, and Calb1 is a calcium-binding protein involved in neuronal function.

To determine the specific spatial clusters or anatomical regions that these genes are mapped to, you would need to consult the original study or data source that generated the graph.


In [6]:
# Query 3: Anti-Hallucination Trap
question3 = "Does the graph indicate any relationship between Tau tangle formation and the FDA-approved drug Lecanemab?"
answer3 = query_graph(agent, question3)
print(answer3)

  Querying agent: Does the graph indicate any relationship between Tau tangle formation and the FD...
There is no information in the provided text about Tau tangle formation or the FDA-approved drug Lecanemab. The text appears to be a list of gene associations with various cell types and anatomical regions, but it does not mention either of these specific topics.
